# baseline v4 — 고도화 버전

`baseline v3`에서 다음을 바꿨습니다. 각 셀 제목의 `[변경]` 표시를 따라가면 됩니다.

| # | 변경 | 이유 |
|---|---|---|
| 1 | **라벨 마스킹** | v3는 프롬프트·이미지 패드 토큰까지 전부 학습했습니다. 정답 글자 1토큰만 학습하도록 수정 |
| 2 | **생성+파싱 → 로그확률 스코어링** | 파싱 실패로 인한 `"a"` 강제 반환 제거, 확률값 확보 |
| 3 | **검증 "정확도" 측정** | v3는 valid loss만 봤습니다. 정확도가 없으면 개선 여부를 알 수 없습니다 |
| 4 | **학습 전 제로샷 점수 측정** | 파인튜닝이 실제로 도움이 되는지 판단할 기준선 |
| 5 | **해상도 상향** (384² → 조절 가능) | 재질 표기·작은 글자 인식 |
| 6 | **EXIF 회전 / NFC 정규화** | 휴대폰 수집 데이터의 조용한 손상 |
| 7 | **선지 셔플 증강 + 순열 TTA** | 선지 위치 편향 제거 |
| 8 | **T4 dtype 정합성 + 그래디언트 클리핑** | T4는 bf16 미지원. v3는 bf16 autocast + fp16 GradScaler 혼용 |
| 9 | **이미지 단위 GroupKFold 검증셋** | 같은 이미지가 train/valid에 갈라지는 누수 차단 |

> 실행 순서는 v3와 동일합니다. 맨 위 `CONFIG` 셀의 값만 바꿔가며 실험하세요.


# 환경 준비

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작


In [ ]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas scikit-learn --upgrade

# [재현성] 대회 최종 제출 시에는 아래 셀 출력의 버전을 requirements.txt에 그대로 고정하세요.
#          transformers를 git main에서 설치하면 날마다 결과가 달라질 수 있습니다.

In [ ]:
import torch, transformers, peft
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("transformers:", transformers.__version__, "| peft:", peft.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("bf16 지원:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

# 데이터 준비

In [ ]:
# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 압축 해제
!unzip -q "/content/drive/My Drive/2026-ssafy-15-2-ai.zip" -d "/content/" 

# 라이브러리, 설정  `[변경]`

**모든 실험 손잡이를 이 셀 하나에 모았습니다.** 아래 값만 바꿔가며 실험하고, 결과를 실험 로그에 기록하세요.


In [ ]:
import os, re, math, random, unicodedata, time, json
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageFile
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Dict, List, Any
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True          # [변경] 손상 이미지 허용

device = "cuda" if torch.cuda.is_available() else "cpu"

# ─────────────────────────── CONFIG ───────────────────────────
MODEL_ID    = "Qwen/Qwen2.5-VL-3B-Instruct"

# [변경 5] 해상도. Qwen-VL은 "픽셀 수"로 지정하며 28*28 이 시각토큰 1개입니다.
#   v3: min=max=384*384(=147k px, 시각토큰 188개) → 라벨 글자가 뭉개집니다.
#   아래는 시각토큰 256~768개. OOM이 나면 MAX_PIXELS부터 낮추세요.
MIN_PIXELS  = 256 * 28 * 28
MAX_PIXELS  = 768 * 28 * 28

# [변경] 학습 규모. v3는 200개/1epoch 고정이었습니다.
N_TRAIN     = None      # None = 전체 사용. 시간이 부족하면 1000, 2000 …
EPOCHS      = 2
LR          = 1e-4
GRAD_ACCUM  = 4
LORA_R      = 16        # v3: 8
LORA_ALPHA  = 32
SHUFFLE_CHOICES_P = 0.5 # [변경 7] 학습 시 선지 순서를 섞을 확률 (위치 편향 방지)

# [변경 2·7] 추론 설정
INFER_BS    = 4         # 배치 추론. OOM 나면 1~2로
TTA_ROUNDS  = 4         # 선지 순열 TTA 횟수 (1 = 끄기, 4 = 전체 회전)
VALID_MAX   = 300       # 검증 정확도 측정에 쓸 최대 샘플 수 (속도 조절)

SEED = 42
# ──────────────────────────────────────────────────────────────

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()

# [변경 8] T4(Turing)는 bf16을 지원하지 않습니다. 하드웨어에 맞춰 자동 선택.
USE_BF16   = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE  = torch.bfloat16 if USE_BF16 else torch.float16
print("Device:", device, "| AMP dtype:", AMP_DTYPE)

train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")
print("train:", train_df.shape, "| test:", test_df.shape)

KEYS = ["a", "b", "c", "d"]

# EDA — 데이터를 먼저 의심하기  `[신규]`

**모델을 건드리기 전에 이 셀을 반드시 한 번 읽으세요.** 여기서 나온 값이 이후 모든 판단의 근거입니다.


In [ ]:
# 1) 컬럼과 결측
print("컬럼:", list(train_df.columns))
print(train_df.isna().sum()[lambda s: s > 0], "\n")

# 2) 정답 분포 — 한쪽으로 쏠려 있으면 "다수 클래스 예측"이 25%보다 높은 무료 하한선이 됩니다
ans = train_df["answer"].astype(str).str.strip().str.lower()
print("정답 분포:\n", ans.value_counts(normalize=True).round(4), "\n")
print("선지 개수가 4가 아닌 행:", int(train_df[KEYS].isna().any(axis=1).sum()), "\n")

# 3) 이미지당 질문 수 — 1보다 크면 반드시 '이미지 단위'로 split 해야 합니다
img_key = train_df["path"].astype(str).map(os.path.basename)
print("고유 이미지 수:", img_key.nunique(), "/ 행 수:", len(train_df))
print("이미지당 질문 수 분포:\n", img_key.value_counts().value_counts().head(), "\n")

# 4) 경로 유효성 + 해상도/EXIF 표본
bad = [p for p in train_df["path"].head(200) if not os.path.exists(p)]
print("경로가 없는 파일(앞 200개 중):", len(bad))
sizes, rotated = [], 0
for p in train_df["path"].head(50):
    im = Image.open(p)
    sizes.append(im.size)
    if im.getexif().get(274, 1) not in (1, None):   # 274 = Orientation
        rotated += 1
print("해상도 표본:", sizes[:5])
print(f"EXIF 회전이 걸린 이미지: {rotated}/50  ← 0이 아니면 exif_transpose 필수\n")

# 5) 질문 유형 — 100개만 눈으로 보세요. 전처리·증강 판단의 축입니다.
for q in train_df["question"].drop_duplicates().head(15):
    print(" -", q)

# 모델, Processor  `[변경]`

- 해상도를 `MIN_PIXELS`/`MAX_PIXELS`로 분리 (v3는 min=max로 고정 → 종횡비까지 뭉갬)
- 4bit 연산 dtype을 AMP dtype과 일치시킴
- LoRA rank 8 → 16


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=AMP_DTYPE,        # [변경 8] 학습 autocast와 dtype 일치
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,                   # [변경 5] min != max → 종횡비 유지
    max_pixels=MAX_PIXELS,
    trust_remote_code=True,
)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
base_model.config.use_cache = False           # 그래디언트 체크포인팅과 함께 쓸 때 필수

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 프롬프트 · 유틸  `[변경]`

- `load_image`: **EXIF 회전 보정** + RGB 통일
- `norm`: **NFC 정규화** (맥에서 만든 한글 문자열은 자모가 분리된 NFD입니다)
- `ANS_IDS`: `a`/`b`/`c`/`d` 의 **토큰 ID**. 라벨 마스킹과 로그확률 스코어링의 핵심


In [ ]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

def load_image(path):
    """[변경 6] 휴대폰 사진의 EXIF 회전을 보정하지 않으면 사진이 누운 채로 입력됩니다."""
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    return img.convert("RGB")

def norm(s):
    """[변경 6] 한글 NFD → NFC 정규화. 안 하면 문자열 비교와 토크나이징이 어긋납니다."""
    return unicodedata.normalize("NFC", str(s)).strip()

# [변경 1·2] 정답 글자의 토큰 ID — 학습 라벨과 추론 스코어링이 같은 대상을 봅니다
ANS_IDS = {k: processor.tokenizer.encode(k, add_special_tokens=False)[0] for k in KEYS}
ANS_ID_LIST = [ANS_IDS[k] for k in KEYS]
print("ANS_IDS:", ANS_IDS)
assert len(set(ANS_ID_LIST)) == 4, "a~d 토큰 ID가 겹칩니다. 토크나이저를 확인하세요."

def build_messages(image, question, shown, gold=None):
    """shown: 화면에 (a)(b)(c)(d) 순서로 보여줄 선지 4개"""
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": build_mc_prompt(norm(question), *[norm(x) for x in shown])},
        ]},
    ]
    if gold is not None:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": gold}]})
    return msgs

# Custom Dataset, Collator  `[변경 — 가장 중요]`

### v3의 문제
```python
enc["labels"] = enc["input_ids"].clone()      # ← 전체 시퀀스를 학습
```
이렇게 하면 손실이 **시스템 프롬프트 + 질문 + 선지 + 이미지 패드 토큰 수백 개**에 걸쳐 계산됩니다.
정작 맞혀야 할 정답 글자 1토큰의 비중은 1/수백이라 학습 신호가 거의 사라지고,
모델은 `<|image_pad|>`를 예측하도록 학습됩니다.

### v4
정답 글자 토큰 **하나만** 남기고 나머지는 전부 `-100`(무시)으로 마스킹합니다.


In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, train=True, shuffle_p=0.0):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.shuffle_p = shuffle_p          # [변경 7] 선지 순서 셔플 증강 확률

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = load_image(row["path"])
        contents = [row[k] for k in KEYS]
        gold = norm(row["answer"]).lower() if self.train else None

        if self.train and gold in KEYS and random.random() < self.shuffle_p:
            gold_idx = KEYS.index(gold)
            order = random.sample(range(4), 4)          # 위치 i 에 원래 선지 order[i]
            contents = [contents[j] for j in order]
            gold = KEYS[order.index(gold_idx)]          # 정답 글자도 함께 이동

        return {"image": img, "question": row["question"], "shown": contents, "gold": gold}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images, golds = [], [], []
        for s in batch:
            msgs = build_messages(s["image"], s["question"], s["shown"],
                                  gold=s["gold"] if self.train else None)
            texts.append(self.processor.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=not self.train))
            images.append(s["image"])
            golds.append(s["gold"])

        self.processor.tokenizer.padding_side = "right" if self.train else "left"
        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")

        if self.train:
            # [변경 1] 전체를 -100으로 덮고, 정답 글자 토큰 한 개만 되살립니다.
            labels = torch.full_like(enc["input_ids"], -100)
            for bi, g in enumerate(golds):
                ids = enc["input_ids"][bi]
                pos = (ids == ANS_IDS[g]).nonzero(as_tuple=True)[0]
                assert len(pos) > 0, f"정답 토큰을 찾지 못했습니다: {g}"
                labels[bi, pos[-1]] = ANS_IDS[g]        # 마지막 등장 = assistant 응답 위치
            enc["labels"] = labels
        return enc

# 라벨 마스킹 검증  `[신규]`

**이 셀이 통과하지 않으면 학습해봐야 소용없습니다.** 학습 대상 토큰이 정확히 정답 글자 1개인지 눈으로 확인합니다.


In [ ]:
_probe_ds = VQAMCDataset(train_df.head(4), train=True, shuffle_p=0.0)
_probe_batch = DataCollator(processor, train=True)([_probe_ds[i] for i in range(4)])

for bi in range(4):
    ids, lab = _probe_batch["input_ids"][bi], _probe_batch["labels"][bi]
    keep = (lab != -100).nonzero(as_tuple=True)[0]
    print(f"[{bi}] 전체 {len(ids)} 토큰 중 학습 대상 {len(keep)}개 "
          f"→ {processor.tokenizer.decode(ids[keep])!r} "
          f"(정답: {train_df.iloc[bi]['answer']})")
    assert len(keep) == 1, "학습 대상 토큰이 1개가 아닙니다. 마스킹을 확인하세요."
print("\n라벨 마스킹 정상")

# 검증셋 분리  `[변경 9]`

v3: `train_df.sample(200)` 후 앞 90% / 뒤 10% → **검증 20건**. 정확도 오차가 ±10%p라 아무것도 판단할 수 없습니다.
또 같은 이미지가 train과 valid에 갈라지면 점수가 뻥튀기됩니다.

v4: **이미지 단위 GroupKFold + 정답 글자 stratify**, 검증셋을 충분히 크게.


In [ ]:
groups = train_df["path"].astype(str).map(os.path.basename)
y      = train_df["answer"].astype(str).str.strip().str.lower()

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_idx, va_idx = next(sgkf.split(train_df, y, groups))

train_subset = train_df.iloc[tr_idx].reset_index(drop=True)
valid_subset = train_df.iloc[va_idx].reset_index(drop=True)

# 누수 확인 — 반드시 0이어야 합니다
leak = set(groups.iloc[tr_idx]) & set(groups.iloc[va_idx])
print("train/valid 공유 이미지 수:", len(leak))
assert len(leak) == 0

# 검증셋은 속도를 위해 상한만 두고, 학습셋은 시간 예산에 맞춰 조절
valid_eval = valid_subset.sample(n=min(VALID_MAX, len(valid_subset)),
                                 random_state=SEED).reset_index(drop=True)
if N_TRAIN is not None:
    train_subset = train_subset.sample(n=min(N_TRAIN, len(train_subset)),
                                       random_state=SEED).reset_index(drop=True)

print(f"train {len(train_subset)} | valid {len(valid_subset)} (평가에 {len(valid_eval)}개 사용)")

train_ds = VQAMCDataset(train_subset, train=True, shuffle_p=SHUFFLE_CHOICES_P)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                          collate_fn=DataCollator(processor, True), num_workers=0)

# 평가 함수 — 로그확률 스코어링 + 순열 TTA  `[신규 · 변경 2·7]`

### v3의 문제
```python
out_ids = model.generate(..., max_new_tokens=2)
preds.append(extract_choice(output_text))     # 실패하면 무조건 "a"
```
모델이 `"정답은"` 같은 걸 뱉으면 파싱이 실패하고 **전부 `a`로 밀립니다.**

### v4
생성하지 않고, 다음 토큰 분포에서 **`a`/`b`/`c`/`d` 4개의 로그확률만 읽습니다.**
- 파싱 실패가 구조적으로 0
- 확률값이 남아 TTA·앙상블에 재사용 가능
- 생성보다 빠름 (forward 1회)

여기에 **선지 순서를 회전시켜 4번 채점하고 내용 기준으로 확률을 평균**내면 위치 편향이 사라집니다.


In [ ]:
@torch.no_grad()
def score_batch(rows, shown_list):
    """각 행에 대해 (a,b,c,d) 위치별 로그확률 (B,4) 반환"""
    texts, images = [], []
    for row, shown in zip(rows, shown_list):
        img = load_image(row["path"])
        msgs = build_messages(img, row["question"], shown, gold=None)
        texts.append(processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
        images.append(img)

    processor.tokenizer.padding_side = "left"      # left padding → [:, -1] 이 마지막 실토큰
    enc = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)
    with torch.autocast("cuda", dtype=AMP_DTYPE):
        logits = model(**enc).logits[:, -1, :]     # 다음 토큰 분포
    logp = torch.log_softmax(logits.float(), dim=-1)
    return logp[:, ANS_ID_LIST].cpu()


def predict_df(df, tta_rounds=TTA_ROUNDS, batch_size=INFER_BS, desc="predict"):
    """순열 TTA 적용 예측. 반환: (예측 글자 리스트, 원래 선지 기준 확률 (N,4))"""
    was_training = model.training
    model.eval()
    preds, probs = [], []
    for s in tqdm(range(0, len(df), batch_size), desc=desc, unit="batch"):
        chunk = [df.iloc[i] for i in range(s, min(s + batch_size, len(df)))]
        agg = torch.zeros(len(chunk), 4)
        for r in range(tta_rounds):
            perm  = [(i + r) % 4 for i in range(4)]          # 위치 i 에 원래 선지 perm[i]
            shown = [[row[KEYS[perm[i]]] for i in range(4)] for row in chunk]
            p = torch.softmax(score_batch(chunk, shown), dim=-1)
            for i in range(4):
                agg[:, perm[i]] += p[:, i] / tta_rounds      # 원래 선지 기준으로 되돌리기
        preds += [KEYS[j] for j in agg.argmax(dim=-1).tolist()]
        probs.append(agg)
    if was_training:
        model.train()
    return preds, torch.cat(probs)


def evaluate(df, tta_rounds=1, desc="valid"):
    preds, _ = predict_df(df, tta_rounds=tta_rounds, desc=desc)
    gold = [norm(x).lower() for x in df["answer"]]
    return float(np.mean([p == g for p, g in zip(preds, gold)]))

# 학습 전 제로샷 정확도  `[신규 · 변경 4]`

**이 숫자가 진짜 베이스라인입니다.** 파인튜닝 후 점수가 이것보다 낮으면 학습이 잘못된 것이고,
이 숫자가 이미 충분히 높다면 학습보다 프롬프트·해상도·TTA에 시간을 쓰는 편이 낫습니다.

LoRA는 초기화 시 항등 변환이므로 지금 평가하면 원본 모델의 제로샷 성능이 나옵니다.


In [ ]:
t0 = time.time()
zs_acc     = evaluate(valid_eval, tta_rounds=1, desc="zero-shot")
zs_acc_tta = evaluate(valid_eval, tta_rounds=TTA_ROUNDS, desc="zero-shot+TTA")
print(f"\n제로샷            : {zs_acc:.4f}")
print(f"제로샷 + 순열 TTA : {zs_acc_tta:.4f}   (랜덤 = 0.2500)")
print(f"소요: {time.time()-t0:.0f}s")

# 다수 클래스 하한선 — 이보다 낮으면 무언가 잘못된 것입니다
print("다수 클래스 예측:", y.value_counts(normalize=True).max().round(4))

# fine-tuning  `[변경 8]`

- bf16/fp16을 하드웨어에 맞춰 선택, GradScaler는 fp16일 때만 사용
- 그래디언트 클리핑 추가
- 마지막 자투리 배치도 반영
- **에폭마다 valid loss가 아니라 valid 정확도를 측정**하고, 가장 좋은 어댑터만 저장


In [ ]:
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.0)

steps_per_epoch   = math.ceil(len(train_loader) / GRAD_ACCUM)
num_training_steps = EPOCHS * steps_per_epoch
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(num_training_steps * 0.03), num_training_steps)

scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))

SAVE_DIR = "/content/qwen2_5_vl_3b_lora"
best_acc, history = zs_acc, []
t_start = time.time()

model.train()
for epoch in range(EPOCHS):
    running, seen = 0.0, 0
    bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            loss = model(**batch).loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        running += loss.item() * GRAD_ACCUM; seen += 1

        is_last = (step == len(train_loader))
        if step % GRAD_ACCUM == 0 or is_last:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)   # [변경 8] 발산 방지
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            bar.set_postfix({"loss": f"{running/max(seen,1):.3f}"})
            running, seen = 0.0, 0

    # [변경 3] loss가 아니라 정확도로 판단합니다
    acc = evaluate(valid_eval, tta_rounds=1, desc=f"Epoch {epoch+1} [valid]")
    history.append(acc)
    print(f"[Epoch {epoch+1}] valid acc = {acc:.4f}  (제로샷 {zs_acc:.4f})")
    if acc > best_acc:
        best_acc = acc
        model.save_pretrained(SAVE_DIR); processor.save_pretrained(SAVE_DIR)
        print(f"  ↳ 최고 성능 갱신, 저장: {SAVE_DIR}")
    model.train()

train_minutes = (time.time() - t_start) / 60
print(f"\n학습 시간 {train_minutes:.1f}분 | 제로샷 {zs_acc:.4f} → 최고 {best_acc:.4f}")
if best_acc <= zs_acc:
    print("파인튜닝이 제로샷을 넘지 못했습니다. 학습 데이터 수를 늘리거나 LR을 낮춰보세요 (5e-5).")

# 최종 검증 — 제출 설정 그대로  `[신규]`

테스트에 쓸 설정(TTA 포함) 그대로 검증셋을 한 번 더 재서, **제출 전에 점수를 예측**합니다.


In [ ]:
final_acc = evaluate(valid_eval, tta_rounds=TTA_ROUNDS, desc="final valid")
print(f"제출 설정 기준 검증 정확도: {final_acc:.4f}")
print(f"  제로샷 {zs_acc:.4f} → 제로샷+TTA {zs_acc_tta:.4f} → 학습후+TTA {final_acc:.4f}")

# inference  `[변경 2·7]`

`generate` + 문자열 파싱을 로그확률 스코어링 + 순열 TTA로 교체했습니다.
확률 `test_probs`는 나중에 다른 모델과 앙상블할 때 그대로 쓸 수 있으니 저장해 두세요.


In [ ]:
t0 = time.time()
preds, test_probs = predict_df(test_df, tta_rounds=TTA_ROUNDS, desc="Inference")
infer_sec = time.time() - t0

submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False, encoding="utf-8")
np.save("/content/test_probs.npy", test_probs.numpy())     # 앙상블용 확률 보관
print(f"Saved /content/submission.csv  ({infer_sec/60:.1f}분, {infer_sec/len(test_df):.2f}s/장)")

# 제출 파일 검증  `[신규]`

**매번 제출 직전에 실행하세요.** 리더보드가 25% 근처에서 안 움직이는 사고의 대부분이 여기서 걸립니다.


In [ ]:
sub = pd.read_csv("/content/submission.csv")
assert list(sub.columns) == ["id", "answer"],       "컬럼명/순서 불일치"
assert len(sub) == len(test_df),                    "행 수 불일치"
assert sub["id"].tolist() == test_df["id"].tolist(), "id 순서 불일치"
assert sub["answer"].isin(KEYS).all(),              "소문자 a~d 이외의 값 존재"
assert sub["answer"].notna().all(),                 "결측 존재"
print("제출 파일 정상\n")
print(sub["answer"].value_counts(normalize=True).round(4))   # 한쪽으로 쏠렸으면 의심
sub.head()

# 효율성 리포트  `[신규]`

대회 채점 기준의 **"모델의 효율성 및 성능"** 항목입니다. 측정값을 제시하는 팀이 의외로 드물어서
이 표 하나가 그대로 점수가 되고, 발표 자료의 핵심 슬라이드가 됩니다.


In [ ]:
total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in trainable)
peak_gb     = torch.cuda.max_memory_allocated() / 1e9

report = {
    "model": MODEL_ID,
    "quantization": "4bit nf4",
    "visual_tokens": f"{MIN_PIXELS//784}~{MAX_PIXELS//784}",
    "total_params": f"{total_p/1e9:.2f}B",
    "trainable_params(LoRA)": f"{trainable_p/1e6:.1f}M ({trainable_p/total_p*100:.2f}%)",
    "train_samples": len(train_subset),
    "epochs": EPOCHS,
    "train_minutes": round(train_minutes, 1),
    "peak_VRAM_GB": round(peak_gb, 2),
    "tta_rounds": TTA_ROUNDS,
    "infer_sec_per_image": round(infer_sec / len(test_df), 3),
    "valid_acc_zeroshot": round(zs_acc, 4),
    "valid_acc_zeroshot_tta": round(zs_acc_tta, 4),
    "valid_acc_final": round(final_acc, 4),
}
print(json.dumps(report, indent=2, ensure_ascii=False))
with open("/content/efficiency_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# 다음 수 — 여기서 더 올리려면

이 노트북이 돌아간 뒤, **오답 100개를 눈으로 보고** 유형별로 세는 것부터 하세요. 그다음:

| 오답 유형 | 다음 수 |
|---|---|
| 라벨·표기 글자를 못 읽음 | `MAX_PIXELS`를 1280*28*28까지 올리거나, 외부 OCR 결과를 프롬프트에 주입 |
| 물체가 작아서 놓침 | 전체 + 크롭 2뷰 입력 |
| 개수 세기에서 틀림 | 해상도 상향 + 짧은 CoT (이 유형에만 선택적으로) |
| 상식형에서 틀림 | 7B 모델로 교체 (`Qwen2.5-VL-7B-Instruct`) — 언어 백본 크기가 지배적 |
| 특정 선지에 쏠림 | `SHUFFLE_CHOICES_P`를 1.0으로, `TTA_ROUNDS` 유지 |
| 전반적으로 애매 | `test_probs.npy`를 모델별로 모아 확률 평균 앙상블 |

`test_probs.npy`를 실험마다 다른 이름으로 저장해 두면, 나중에 앙상블할 때 재추론이 필요 없습니다.
